In [ ]:
#| eval: false
! [ -e /content ] && pip install -Uqq xcube # upgrade xcube on colab

In [ ]:
from fastai.data.core import *
from xcube.l2r.all import *

Make sure we have that "beast":

In [ ]:
#| eval: false
ic(torch.cuda.get_device_name(default_device()));
test_eq(torch.cuda.get_device_name(0), torch.cuda.get_device_name(default_device()))
test_eq(default_device(), torch.device(0))
print(f"GPU memory = {torch.cuda.get_device_properties(default_device()).total_memory/1024**3}GB")

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

Setting some environment variables:

In [ ]:
# os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

Setting defaults for pandas and matplotlib:

In [ ]:
# Set the default figure size
plt.rcParams["figure.figsize"] = (8, 4)
# Set pandas column width
pd.set_option('display.max_colwidth', None)

Altering some default jupyter settings:

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "last" # "all"

# Boot L2R 

> Bootstrapping a learning-to-rank model

In this tutorial we will find a needle in the haystack with mutual infomation gain:

#### Mutual-Information Computation

In [ ]:
source = untar_xxx(XURLs.MIMIC3_L2R)

In [ ]:
data = source/'mimic3-9k.csv'
df = pd.read_csv(data,
                 header=0,
                 names=['subject_id', 'hadm_id', 'text', 'labels', 'length', 'is_valid'],
                 dtype={'subject_id': str, 'hadm_id': str, 'text': str, 'labels': str, 'length': np.int64, 'is_valid': bool})
df[['text', 'labels']] = df[['text', 'labels']].astype(str)
len(df)

In [ ]:
df.head(3)

The file `'code_desc.pkl'` contains a short description for the labels. 

In [ ]:
# with open(source/'code_desc.pkl', 'rb') as f: lbs_desc = pickle.load(f)
lbs_desc = load_pickle(source/'code_desc.pkl')
assert isinstance(lbs_desc, dict)
test_eq(mapt(lbs_desc.get, ['00.93', '427.31', '00.09']), ('Transplant from cadaver',
                                                          'Atrial fibrillation',
                                                          'Other therapeutic ultrasound'))

Note that performing some computations in this notebook on the full dataset is going to take a lot of time. But don't worry `untar_xxx` has already downloaded everything you need. But you can still run the following cells if you want to generate everything from scratch. Preferably, run the following cells on a sampled dataset for quick iterations. 

**Run the cell below only if you want to sample from the full dataset to create a tiny dataset for the purpose of quick iterations.**

*Technical Point:* If we want to sample to perform quick iterations, we need to make sure the number of data points in the sample is a multiple of `bs`. So that we do not have to do a `drop_last=True` while creating the `Dataloaders`. This is because we are about to do some probability computations, and dropping data points is not a good idea as probabilities would not sum to 1.

In [ ]:
bs = 8
cut = len(df) - len(df)%bs
df = df[:cut]
len(df)

In [ ]:
_arr = np.arange(0, len(df), bs)
# mask = (_arr > 4000) & (_arr < 5000)
mask = (_arr > 500) & (_arr < 1000)
_n = np.random.choice(_arr[mask], 1)
df = df.sample(n=_n, random_state=89, ignore_index=True)
len(df)

In [ ]:
df.head(3)

**[Mutual Information](https://en.wikipedia.org/wiki/Mutual_information#)**

<img alt="Pictorial representation of simple neural network" width="400" src="pics/info-gain.svg" caption="Pictorial representation of mutual information gain" id="img_mut_info">

The mutual information of two jointly discrete random variables X and  Y is calculated as a double sum:

$$I(T;L) = \sum_{l \in \mathcal{L}} \sum_{t in \mathcal{T}} P_{(T,L)}(t,l) \log \Bigg(\frac{P_{(T,L)}(t,l)}{P_T(t) P_L(l)} \Bigg)$$

where $P_{(T,L)}$ is the [joint probability mass function](https://en.wikipedia.org/wiki/Joint_distribution) of $T$ and $L$, and $P_T$ and $P_L$ are the [marginal probability mass fucntions](https://en.wikipedia.org/wiki/Marginal_probability) of $T$ and $L$ respectively. To compute $I$, the only quantity we need to compute is the joint pmf $P_{(T,L)}$, as the marginal pmfs can be computed from the joint pmf.

With regard to implementation, $P_{(T,L)}$ can be thought of as a 2x2 tensor as shown below:

In [ ]:
p_TL = pd.DataFrame(0, columns=['t', 'not t'], index=['lbl', 'not lbl'])
p_TL

...and we need to compute this $P_{(T,L)}$ for every token-label pair. In other words, we need to fill in the `joint` dataframe shown below. Note that each cell in `joint` dataframe can be thought of to be further subdivided into a 2x2 grid containing the corresponding `p_TL`.

In [ ]:
bs, chnk_sz = 8, 200
info = MutualInfoGain(df, bs=bs, chnk_sz=chnk_sz, lbs_desc=source/'code_desc.pkl') # provide lbs_desc if you have it

In [ ]:
%%time
dsets = info.onehotify()

In [ ]:
toks, lbs = dsets.vocab
L(toks), L(lbs), len(toks)*len(lbs)

In [ ]:
joint = pd.DataFrame(0, columns=range(len(lbs)), index=range(len(toks)))
joint.index.name = 'toks (T)'
joint.columns.name = 'lbs (L)'
joint

We can perform tensorized computation if we think of `p_TL` as a 4 dim tensor of size `(len(toks), len(lbs), 2, 2)`. Next, to be able to estimate `p_TL` we just need to iterate over the dataset and for each data point and each token-label pair record the `p_TL` information in the last two dimension of the tensor `p_TL`. And, at the end divide by size of the dataset. 

Some more implementation details (Skip this if not iterested): 

- We are going to one-hot encode the dataset (both `text` and `labels` field in the `df`). This is done by `onehot_dsets` 
- For efficieny, in reality we are not going to iterate over the dataset one by one, instead we are going to use a dataloader and perform `p_TL` computation on a mini-batch.
- Unless you are doing this in 2035 you probably do not have enogh GPU-RAM to fit the entire `p_TL` tensor of dimension `(len(toks), len(lbs), 2, 2)`. So we are going to split the lbs dimension into chunks. (Why the `lbs` dimension and not the `toks`? Because in XML datsets `toks` are approximately 60000, but the number of `lbs` could be really large of the order of millions.) With reagrd to implementation this would mean that instead of one dataloader we would roll with multiple dataloaders. And each dataloader would load the dataset in a way that mini-batches would contain the full one-hot encoding of the `text` field but only a certain `chunk` of the one-hot encoded `labels` field in `df`. Another way to think about this is that each datapoint, specifically the `labels` are splitted across multiple dataloaders. This way once we are done iterating over one such dataloader we would have filled a ceratin chunk of the `joint` dataframe shown above. And we would fill the entire `joint` only once we are done iterating over all the dataloaders. 

In [ ]:
x, y = dsets[0]
test_eq(tensor(dsets.tfms[1][2].decode(y)), torch.where(y==1)[0])
test_eq(tensor(dsets.tfms[0][-1].decode(x)), torch.where(x==1)[0])

In [ ]:
' '.join(L(toks)[torch.where(x==1)[0]])

In [ ]:
lbs.map_ids(torch.where(y==1)[0])

In [ ]:
#| hide
#| eval: false
from fastai.data.transforms import *
from fastai.data.core import *
from fastai.text.core import *

In [ ]:
#| hide
#| eval: false
splits = ColSplitter()(df)
splits

# lm_vocab = torch.load(dls_lm_vocab_path)

@Transform
def Cleanser(toks): return [o for o in toks if o in lm_vocab]

class MyNumericalize(Transform):
    "Transform to remove tokens not present in `vocab`"
    def __init__(self, vocab=None, min_freq=3, max_vocab=60000, special_toks=None):
        store_attr('vocab,min_freq,max_vocab,special_toks')
        self.o2i = None if vocab is None else defaultdict(int, {v: i for i,v in enumerate(vocab)})
    
    def setups(self, dsets):
        if dsets is None: return
        if self.vocab is None:
            count = dsets.counter if getattr(dsets, 'counter', None) is not None else Counter(p for o in dsets for p in o)
            if self.special_toks is None and hasattr(dsets, 'special_toks'):
                self.special_toks = dsets.special_toks
            self.vocab = make_vocab(count, min_freq=self.min_freq, max_vocab=self.max_vocab, special_toks=self.special_toks)
            self.o2i = defaultdict(int, {v:i for i,v in enumerate(self.vocab) if v != 'xxfake'})
    
    def encodes(self, o): return TensorText(tensor([self.o2i[o_] for o_ in o if o_ in self.vocab]))
    def decodes(self, o): return L(self.vocab[o_] for o_ in o)

# resort to this if anythiong goes wrong below
x_tfms = [Tokenizer.from_df('text', n_workers=num_cpus()), attrgetter("text"), Cleanser, MultiCategorize(vocab=lm_vocab), OneHotEncode()]
y_tfms = [ColReader('labels', label_delim=';'), MultiCategorize(), OneHotEncode()]
tfms = [x_tfms, y_tfms]

class Chunkifize(Transform):
    order = 4
    def __init__(self, num_chunks=3): store_attr('num_chunks')
    def encodes(self, o): 
        return list(torch.chunk(o, self.num_chunks))
    def decodes(self, o): 
        return torch.cat(o)

chnk_tfm = Chunkifize()
chnks = chnk_tfm(torch.arange(10))
test_eq(type(chnks), list)
test_eq(chnks, [tensor([0, 1, 2, 3]), tensor([4, 5, 6, 7]), tensor([8, 9])])
# test_fail(lambda: chnk_tfm.decode(chnks), tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]))
test_eq(chnk_tfm.decode(chnks), tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]))

# y_tfms = [ColReader('labels', label_delim=';'), MultiCategorize(), OneHotEncode(), Chunkifize()]
# tfmd_y = TfmdLists(df, tfms=y_tfms)
# tfmd_y.decode(tfmd_y[0])

In [ ]:
dls = info.lbs_chunked()

In [ ]:
assert isinstance(dls[0], TfmdDL)
test_eq(len(dls),  np.ceil(len(lbs)/200))
test_eq(len(dls[0]), np.ceil(len(dsets)/bs)) # drop_last is False
# test to prove that the labels for each data point is split across multiple dataloaders
lbs_0 = torch.cat([yb[0] for dl in dls for _,yb in itertools.islice(dl, 1)])
y = y.to(default_device())
test_eq(lbs_0, y)

Now let's compute the `joint_pmf` table we had seen earlier. 

In [ ]:
%%time
p_TL = info.joint_pmf()

In [ ]:
test_eq(p_TL.shape, (info.toksize, info.lblsize, 2, 2))

Technicality: `p_TL` is not really the joint pmf (yes, I lied before!) but contains all the information needed to compute the joint pmf `p_TxL` and mutual info gain `I_TL`. This computation is going to be comnputed by `compute`:

In [ ]:
%%time
p_T, p_L, p_TxL, H_T, H_L, I_TL = info.compute()

All this while if you have been working with the sampled dataset you can continue to do so for the rest of this notebook. But if you want a real feel of how things look, at this point you can load the pregenerated `p_TL` and `(p_T, p_L, p_TxL, H_T, H_L, I_TL)` for the full dataset which `untar_xxx` downloaded:

In [ ]:
# print('\n'.join(L(source.glob("**/*.pkl")).map(str)))
# or better yet
!tree -sh -P "*.pkl" {source}

In [ ]:
#| eval:false
# %%time 
p_TL = torch.load(source/'p_TL.pkl', map_location=torch.device('cpu'))
p_T, p_L, p_TxL, H_T, H_L, I_TL = torch.load(source/'info.pkl', map_location=torch.device('cpu'))

Make sure there aren't any of those pesky nans or negs:

In [ ]:
def test_nanegs(*args):
    for o in args:
        has_nans = o.isnan().all() # check for nans
        has_negs = not torch.where(o>=0, True, False).all()
        if has_nans: raise Exception(f"{namestr(o, globals())[0]} has nans")
        if has_negs: raise Exception(f"{namestr(o, globals())[0]} has negs")

In [ ]:
test_fail(test_nanegs, args=(p_T, p_L, p_TxL, H_T, H_L, I_TL), contains='I_TL has negs')

Theoretically, Mutual-Info as defined [here](https://en.wikipedia.org/wiki/Mutual_information) is suposed to be non-negative (can be proved by tossing in [Jensen](https://en.wikipedia.org/wiki/Jensen%27s_inequality)). But, practically, it turns out `I_TL` has some negs because we distorted the `p_TL` and `p_TxL`  with `eps` in the `I_TL` computation.

In [ ]:
torch.topk(I_TL.flatten(), 10, largest=False)

In [ ]:
howmany = torch.where(I_TL < 0, True, False).sum().item()
negs = torch.where(I_TL < 0, I_TL, I_TL.new_zeros(I_TL.shape))
negs.sum()/howmany

Those negs on an avg are pretty close to zero. So we need not worry. Let's roll!

In [ ]:
test_eq(p_TL.shape, (info.toksize, info.lblsize, 2, 2))
test_eq(p_T.shape, (info.toksize, 2, 1))
test_eq(p_L.shape, (info.lblsize, 1, 2))
test_eq(p_TxL.shape, (info.toksize, info.lblsize, 2, 2))
test_eq(H_T.shape, [info.toksize])
test_eq(H_L.shape, [info.lblsize])
test_eq(I_TL.shape, (info.toksize, info.lblsize))

In [ ]:
#| hide
# r_t, r_l = random.randrange(0, len(toks)), random.randrange(0, len(lbs))
# toks[r_t], lbs[r_l]

# test_close(p_TL[r_t,r_l].sum(), 1, eps=1e-1)
# test_eq(p_T[r_t].sum(), 1)
# test_eq(p_L[r_l].sum(), 1)

# p_TL[r_t,r_l].sum(-1), p_TL[r_t, 400].sum(-1) 

# p_T[r_t], p_L[r_l]
# I_TL[r_t,r_l]

In [ ]:
eps = I_TL.new_empty(1).fill_(1e-15)
info_lbl_entropy = I_TL/(H_L + eps)
info_jaccard = I_TL/(H_T.unsqueeze(-1) + H_L.unsqueeze(0) - I_TL + eps)
assert not info_lbl_entropy.isnan().all(); assert not info_jaccard.isnan().all()
l2r_bootstrap = {'toks': toks, 'lbs': lbs, 'mut_info_lbl_entropy': info_lbl_entropy, 'mutual_info_jaccard': info_jaccard}

`l2r_bootstrap` for the full dataset was downloaded by `untar_xxx` in `boot_path`. You can load it up in the following cell. `l2r_bootstrap` will be used to bootstrap our learning-to-rank model.

#### Save those Mutual Information Gain values

In [ ]:
# l2r_bootstrap = torch.load(boot_path)

In [ ]:
# l2r_bootstrap['info_jaccard'] = l2r_bootstrap.pop('mutual_info_jaccard')

In [ ]:
# globals().update(l2r_bootstrap)

In [ ]:
# info_jaccard.shape

Let's take a look at the *Mutual Information Gain* (`I_TL`) for each of the labels:

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    args = (p_TL, p_T, p_L, info_jaccard, H_T, H_L)
    kwargs = {'k':10, 'save_as': Path(tmpdir)/'mut_info_jaccard.ft'}
    df_info = info.show(*args, **kwargs)
    assert (Path(tmpdir)/'mut_info_jaccard.ft').exists()

In [ ]:
df_info.head()

#### Let's look at those Mutual-Information Gain values:

In [ ]:
mask = (df_info.freq>50) & (df_info.freq<150)
# with pd.option_context('display.max_colwidth', 100):
# pd.reset_option('all')
df_info = df_info[mask].reset_index(drop=True)
len(df_info)

The dataframe below shows the top 10 tokens (based on the mutual-info-gain values) for labels are rare (freq between 50 and 150). Feel free to ChatGPT the label descriptions and the tokens to find out if we're able to find the needle in a haystack. 

In [ ]:
pd.set_option('display.max_colwidth', None)
df_info.head()

In [ ]:
# pd.reset_option('all')

In [ ]:
# df_info.to_excel('jaccard.xls', index=False)

#### Scratchpad

In [ ]:
from fastai.data.transforms import *

In [ ]:
source = untar_xxx(XURLs.MIMIC3_L2R)
boot_path = source/'mimic3-9k_tok_lbl_info.pkl'
assert boot_path.exists()

In [ ]:
l2r_bootstrap = torch.load(boot_path)
toks, lbs, info = mapt(l2r_bootstrap.get, ['toks', 'lbs', 'mutual_info_jaccard'])
print(toks[-2:]) # last two places has 'xxfake'
# toks = CategoryMap(toks, sort=False)
lbs_des = load_pickle(source/'code_desc.pkl')
assert isinstance(lbs_des, dict)
test_eq(info.shape, (len(toks), len(lbs))) # last two places has 'xxfake'

In [ ]:
L(toks[89:99]), lbs[:10]

In [ ]:
icd_codes = ['008.45', '009.0', '244.9', '250.00']

In [ ]:
d= dict(zip(icd_codes, mapt(lbs_des.get, icd_codes)))
df_des = pd.DataFrame(d, index=range(1))

lbs_idxs = lbs.map_objs(icd_codes)
top_infos = info[:, lbs_idxs].topk(dim=0, k=10, largest=True).values.cpu()
top_idxs = info[:, lbs_idxs].topk(dim=0, k=10, largest=True).indices.cpu().long()
toks = array(toks).astype(str)
# toks.map_ids(tok_idxs[:, 0])
df_toks = pd.DataFrame(toks[top_idxs], columns=icd_codes)
df_infos = pd.DataFrame(top_infos, columns=icd_codes)

In [ ]:
display(df_des, df_toks, df_infos)

In [ ]:
vocab = load_pickle(Path.cwd()/'tmp/models/mimic3-9k_clas_full_vocab.pkl')

In [ ]:
toks = L(toks, use_list=True)

The tokens which are there in the xml vocab but we do not have any 'info' on:

In [ ]:
not_found = L(set(vocab[0]).difference(set(toks)))
not_found

In [ ]:
test_fail(lambda : toks.index('unrmarkable'), contains='is not in list')

The tokens which we have info about but were not present in the xml vocab

In [ ]:
set(toks).difference(vocab[0])

Thankfully, we have `info` for all the labels in the xml vocab:

In [ ]:
test_shuffled(vocab[1], lbs)

Now we need to create a mapping between the indices of the xml vocab and the information gain vocab:

In [ ]:
def xml2info(xml_vocab, info_vocab):
    "Creates a mapping between the indices of the xml vocab and the information-gain vocab"
    xml2info = {i: info_vocab.index(o) if o in info_vocab else np.inf  for i,o in enumerate(xml_vocab)}
    xml2info_notfnd = [o for o in xml2info if xml2info[o] is np.inf]
    return xml2info, xml2info_notfnd

In [ ]:
toks_xml2info, toks_notfnd = xml2info(vocab[0], toks)

In [ ]:
toks_found = set(toks_xml2info).difference(set(toks_notfnd))
test_shuffled(array(vocab[0])[toks_notfnd], not_found)
some_xml_idxs = np.random.choice(array(L(toks_found)), size=10)
some_xml_toks = array(vocab[0])[some_xml_idxs]
corres_info_idxs = L(map(toks_xml2info.get, some_xml_idxs))
corres_info_toks = array(toks)[corres_info_idxs]
assert all_equal(some_xml_toks, corres_info_toks)

In [ ]:
lbs_xml2info, lbs_notfnd = xml2info(L(vocab[1]), L(lbs))

In [ ]:
lbs_found = set(lbs_xml2info).difference(set(lbs_notfnd))
some_xml_idxs = np.random.choice(array(L(lbs_found)), size=10)
some_xml_lbs = array(vocab[1])[some_xml_idxs]
corres_info_idxs = L(map(lbs_xml2info.get, some_xml_idxs))
corres_info_lbs = array(lbs)[corres_info_idxs]
assert all_equal(some_xml_lbs, corres_info_lbs)